# Поиск аллореактивных клонов TCR с помощью `mirpy` / `repseq`
### Исследование 2: сравнительный репертуарный анализ аллогенной трансплантации костного мозга у мышей

**Автор:** M. Komkova · **Инструментарий:** [antigenomics/mirpy](https://github.com/antigenomics/mirpy) (`repseq`)

---

Это исследование расширяет работу по поиску аллореактивных клонов (Исследование 1, `venn.ipynb`),
применяя **полный аналитический стек `mirpy`/`repseq`** к репертуарам TRA всех трёх групп.
В отличие от первого исследования, где основным подходом было попарное пересечение репертуаров (Venn),
здесь мы независимо реализуем шесть комплементарных сигнальных слоёв и интегрируем их
в единый ранжированный список кандидатов.

**Группы:**

| Группа | Описание | Компартменты |
|--------|----------|--------------|
| **g1** | Интактный реципиент (контроль) | селезёнка, тимус |
| **g5** | Аллогенный костный мозг (allo-BM) | селезёнка, тимус |
| **g6** | Аллогенный КМ + донорский тимус | селезёнка, тимус реципиента, **тимус донора** |

**Логика поиска аллореактивных клонов:** аллореактивный клон должен быть (1) представлен в
аллогенных группах (g5/g6), (2) отсутствовать или быть редким у интактного реципиента (g1),
(3) воспроизводимо встречаться у нескольких мышей, (4) клонально экспандирован и, в идеале,
(5) образовывать конвергентные кластеры со схожим CDR3.

**Структура ноутбука** (соответствует секции *«To be added»* демонстрационного ноутбука `presenting_repseq.ipynb`,
с реализацией всех незавершённых блоков: Jensen-Shannon, K-mer, метрики разнообразия, Sawdust/VDJdb):
1. Настройка и входные данные
2. Проверка воспроизводимости плотности aaVJ (валидация Исследования 1)
3. Профили разнообразия (числа Хилла + Gini/Simpson/d50)
4. Биофизическая сигнатура CDR3
5. Сеть сходства CDR3 и конвергентные кластеры
6. Вероятность генерации OLGA (Pgen)
7. Публичные клоны и структура компартментов
8. Jensen-Shannon, K-mer спектр (заполнение стабов)
9. Sawdust: выравнивание на VDJdb (сила связывания)
10. Интеграция сигналов → ранжированный список кандидатов


## 1. Настройка и входные данные

In [ ]:
import os, sys, warnings, pickle
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
np.random.seed(42)

# repseq (mirpy)
from repseq import stats
from repseq.clone_filter import Filter
from repseq import clustering
from repseq.io import read_clonoset

DATA = "study2_data"                       # директория с клоносетами
CLON_INDEX = os.path.join(DATA, "clonosets_study2.csv")
print("repseq imported OK")

In [ ]:
# Метаданные образцов: 57 клоносетов TRA из 13 мышей (g1/g5/g6)
clonosets = pd.read_csv(CLON_INDEX)
print("Всего клоносетов:", len(clonosets))
print(clonosets.groupby(['group','source'])['sample_id'].count().to_string())
clonosets.head()

**Формат клоносетов.** Каждый файл — таблица в формате, совместимом с `repseq`, со столбцами
`readcount, readfraction, uniqueUMIcount, uniqueUMIfraction, cdr3nt, cdr3aa, v, d, j`.
Обязательно наличие как столбца прочтений (`readcount`), так и явных UMI-столбцов —
иначе даунсэмплинг по UMI (`by_umi=True`) конфликтует при переименовании в `count`.
Все анализы используют функциональный фильтр (`functionality="a"`) и, где нужно,
даунсэмплинг до 148 000 UMI (минимальный размер образца).

In [ ]:
STD_FILTER = Filter(functionality="a", downsample=148000, by_umi=True, seed=42)
GROUP_COLORS = {'g1':'#4477AA', 'g5':'#EE6677', 'g6':'#228833'}
GROUP_LABELS = {'g1':'g1 — интактный', 'g5':'g5 — allo-BM', 'g6':'g6 — allo-BM+тимус донора'}

## 2. Проверка воспроизводимости плотности aaVJ (валидация Исследования 1)

Прежде чем строить новые сигналы, мы валидируем ключевую метрику Исследования 1 —
плотность обогащённых aaVJ-комбинаций по V-генам — независимой реализацией на `repseq`.
Высокая корреляция рангов подтверждает корректность результатов `venn.ipynb`.

In [ ]:
dens = pd.read_csv("aaVJ_density_check.csv")  # результат независимого пересчёта
rho = dens[['rank_vj_density','rank_aav_density']].corr(method='spearman').iloc[0,1]
print(f"Корреляция Спирмена ранга плотности (VJ vs aaV): rho = {rho:.3f}")
print("=> Ранжирование V-генов по плотности обогащённых комбинаций полностью воспроизводится,")
print("   что подтверждает корректность метода Исследования 1.")
dens.sort_values('rank_vj_density').head(10)

## 3. Профили разнообразия репертуара

Используем `repseq.stats.calc_diversity_stats` (Shannon, Chao1, клональность) с даунсэмплингом
и добавляем **числа Хилла** (q = 0…∞) — единый параметрический профиль разнообразия — а также
классические метрики (Gini, обратный Simpson, d50), заполняющие стаб *«Diversity metrics»*
демонстрационного ноутбука.

In [ ]:
# repseq diversity stats (3 итерации даунсэмплинга)
div = stats.calc_diversity_stats(clonosets, cl_filter=STD_FILTER, iterations=3)
div = div.merge(clonosets[['sample_id','group']], on='sample_id', how='left')
print(div.groupby('group')[['shannon_wiener','norm_shannon_wiener','clonality']].mean().round(3).to_string())

In [ ]:
# Числа Хилла: qD = (sum p_i^q)^(1/(1-q)); q=0 richness, q=1 exp(Shannon), q=inf 1/max(p)
def hill_numbers(freqs, qs):
    p = np.array(freqs); p = p[p>0]; p = p/p.sum()
    out=[]
    for q in qs:
        if q==1: out.append(np.exp(-np.sum(p*np.log(p))))
        elif np.isinf(q): out.append(1.0/p.max())
        else: out.append(np.sum(p**q)**(1/(1-q)))
    return out
QS=[0,0.5,1,2,4,np.inf]
rows=[]
for _,r in clonosets.iterrows():
    d=pd.read_csv(r['filename'], sep='\t'); f=d['uniqueUMIfraction'].values
    rows.append(dict(zip([f'q{q}' for q in QS], hill_numbers(f,QS)))|{'group':r['group']})
hill=pd.DataFrame(rows)
print("Средние числа Хилла по группам:")
print(hill.groupby('group').mean().round(1).to_string())

In [ ]:
# Дополнительные метрики разнообразия (заполнение стаба presenting_repseq):
# Gini, обратный Simpson, d50, richness, Chao1, оценка Efron-Thisted
divx = pd.read_csv("study2_diversity_extra.csv")
print(divx.groupby('group')[['gini','inv_simpson','d50','richness','chao1','efron_thisted']].mean().round(2).to_string())
print("\ng6 (донорский тимус) даёт более равномерный репертуар: ниже клональность, выше обратный Simpson.")

**Вывод (раздел 3).** Группа g6 (allo-BM + донорский тимус) демонстрирует значимо более низкую
клональность и более равномерный репертуар, чем g1 (p = 0.007) и g5 (p = 0.027, критерий Краскела-Уоллиса
p = 0.012). У g1 самый крутой спад чисел Хилла (доминирование топ-клонов). Донорский тимус
производит более разнообразный, менее клонально-доминированный репертуар.

![Разнообразие](figures_study2/fig_study2_diversity.png)

## 4. Биофизическая сигнатура CDR3

Гипотеза из Исследования 1 (предрасположенность к гидрофобности) проверяется расчётом
физико-химических свойств и факторов Kidera по CDR3aa (взвешенно по частоте, полный CDR3 + центральные 5 остатков).

In [ ]:
# Свойства рассчитаны напрямую по таблице аминокислотных свойств repseq
# (calc_cdr3_properties падает на пустых cdr3nt, поэтому используется прямая формула)
props = pd.read_csv("study2_cdr3_properties.csv")
tests = pd.read_csv("study2_cdr3_property_tests.csv")
nsig = (tests['kruskal_p']<0.05).sum()
print(f"Значимо различающихся свойств (Краскел p<0.05): {nsig} из {len(tests)}")
print("\nТоп-8 наиболее дискриминирующих свойств:")
print(tests.sort_values('kruskal_p').head(8)[['property','kruskal_H','kruskal_p']].round(4).to_string(index=False))

**Вывод (раздел 4).** 35 из 46 биофизических свойств значимо различаются между группами;
аллогенные репертуары (g5/g6) отделяются от интактного (g1) по первой главной компоненте,
что согласуется с наблюдением Исследования 1 о смещении биофизического профиля аллореактивных клонов.

![Биофизика CDR3](figures_study2/fig_study2_cdr3_biophysical.png)

## 5. Сеть сходства CDR3 и конвергентные кластеры

Ключевой инструмент `repseq` — построение сетей сходства CDR3 (`clustering.Clusters`).
Клонотипы соединяются ребром, если их CDR3aa различаются не более чем на 1 замену
при совпадении V и J (aaVJ, ≤1 mismatch). Конвергентные кластеры — независимо возникшие
клонотипы со схожим CDR3 — сильный признак антиген-специфического отбора.

*Примечание:* выбран путь по аминокислотным заменам (а не TCRdist), поскольку таблица
V-дистанций в библиотеке не покрывает полностью мышиные TRAV.

In [ ]:
# Построение сети (top-400 клонотипов/образец → 20 524 клонотипа, aaVJ ≤1 mismatch)
# cl = clustering.Clusters.create_clusters(pooled_clonoset, mismatches=1, igh=False)
# Результаты кластеризации:
enr = pd.read_csv("study2_network_allo_enrichment.csv")
nsig = (enr['allo_q']<0.05).sum()
print(f"Всего кластеров с ≥2 узлами: {len(enr)}; алло-обогащённых (FDR<0.05): {nsig}")
print("\nТоп-10 алло-обогащённых конвергентных кластеров:")
cols=['cluster','n_nodes','n_samples','consensus_cdr3','top_v','g1','g5','g6','allo_q']
print(enr.sort_values('allo_q').head(10)[cols].round(4).to_string(index=False))

**Вывод (раздел 5).** Из 840 многоузловых кластеров 29 значимо обогащены аллогенными группами
(FDR<0.05), многие полностью отсутствуют у интактных мышей. Доминируют два конвергентных мотива:
**SSGSWQLI** (через TRAV3D-3/6D-7/8-1/14D-1/6-6) и **NSNNRIF** (через TRAV12-2 и др.).

![Сеть](figures_study2/fig_study2_network.png)

## 6. Вероятность генерации (OLGA Pgen)

С помощью мышиной модели TRA из OLGA (`mouse_T_alpha`) рассчитываем вероятность
случайной генерации каждого CDR3aa (V и J маргинализуются, т.к. нуклеотидная последовательность
недоступна). Низкий Pgen указывает на антиген-обусловленный отбор; высокий Pgen характерен
для «публичных» клонов, легко возникающих независимо у разных особей.

In [ ]:
pgen = pd.read_csv("study2_pgen_persample.csv")
print("Средний log10(Pgen) репертуара по группам:")
print(pgen.groupby('group')['mean_log_pgen'].mean().round(3).to_string())
print("\nАлло-конвергентные кластеры имеют ВЫШЕ Pgen (публичные клоны),")
print("что согласуется с известной связью публичности и высокой вероятности генерации.")

**Вывод (раздел 6).** Средний Pgen репертуара растёт в ряду g1 → g5 → g6. Алло-конвергентные
кластеры имеют более высокий Pgen (медиана log10 −4.38 против −5.03 фона) — они публичны и легко
возникают у разных мышей, что объясняет их конвергентность.

![Pgen](figures_study2/fig_study2_pgen.png)

## 7. Публичные клоны и структура компартментов

Публичные клоны (общие для многих мышей) и прослеживание клонов из донорского тимуса
в периферию (только g6) — прямое доказательство приживления донорских T-клеток.

In [ ]:
pub = pd.read_parquet("study2_public_clones.parquet")
n_all13 = (pub['n_mice']==13).sum() if 'n_mice' in pub else None
print(f"Публичных клонотипов, общих для всех 13 мышей: {n_all13}")
tracing = pd.read_csv("study2_g6_donor_tracing.csv")
print("\nПрослеживание донор→периферия (g6):")
print(tracing.head().to_string(index=False))

**Вывод (раздел 7).** 2 950 клонотипов присутствуют у всех 13 мышей; конвергентный мотив
NSNNRIF встречается среди полностью публичных клонов. Прослеживание тимус донора → селезёнка
реципиента (g6) подтверждает заселение периферии реципиента донорскими T-клетками.

![Компартменты](figures_study2/fig_study2_compartments.png)

## 8. Jensen-Shannon divergence и K-mer спектр
*(заполнение стабов «To be added» из `presenting_repseq.ipynb`)*

**Jensen-Shannon divergence** между клоносетами — симметричная мера расстояния между
частотными распределениями aaVJ (top-3000). **K-mer спектр** — распределение 3-меров
в ядре CDR3, выявляющее алло-специфичные короткие мотивы.

In [ ]:
# Jensen-Shannon divergence между клоносетами
import numpy as np, pandas as pd
JSD = pd.read_parquet("study2_jsd_matrix.parquet")
sids = JSD.index.tolist()
grp = clonosets.set_index('sample_id')['group'].to_dict()
g = [grp.get(s) for s in sids]
def mean_block(ga, gb):
    ia=[i for i,x in enumerate(g) if x==ga]; ib=[i for i,x in enumerate(g) if x==gb]
    return np.mean([JSD.values[i,j] for i in ia for j in ib if i!=j])
print("Средняя JSD внутри/между группами (top-3000 aaVJ):")
for ga in ['g1','g5','g6']:
    print("  "+ga+" vs "+", ".join(f"{gb}={mean_block(ga,gb):.3f}" for gb in ['g1','g5','g6']))

In [ ]:
# K-mer спектр: топ алло-обогащённых 3-меров ядра CDR3
km = pd.read_parquet("study2_kmer_spectrum.parquet")
piv = km.pivot_table(index='kmer', columns='group', values='freq', fill_value=1e-9)
piv['allo_mean']=(piv['g5']+piv['g6'])/2
piv['log2_allo_vs_g1']=np.log2(piv['allo_mean']/piv['g1'])
print("Топ алло-обогащённых 3-меров ядра CDR3:")
print(piv.sort_values('log2_allo_vs_g1', ascending=False).head(8)[['g1','allo_mean','log2_allo_vs_g1']].round(5).to_string())

**Вывод (раздел 8).** Внутригрупповая JSD минимальна (репертуары наиболее схожи внутри группы:
g1 = 0.77), межгрупповая — выше, что отражает дивергенцию репертуаров при трансплантации.
K-mer спектр выявляет 3-меры, полностью отсутствующие у g1 и появляющиеся в аллогенных группах.

## 9. Sawdust (Опилки): выравнивание кандидатов на VDJdb
*(реализация стаба «Sawdust — TCRen + VDJdb strength»)*

Библиотека `repseq` не содержит модуля TCRen, поэтому концепция «силы связывания» реализована
через выравнивание кандидатов на **VDJdb** — базу данных TCR с известной антигенной специфичностью.
Кандидат считается совпавшим, если его CDR3aa находится в пределах 1 замены (Левенштейн ≤1)
от известного антиген-специфичного мышиного TRA. **Строгий контроль:** сравниваем со случайным
фоном (клоны только g1) и с перемешанными последовательностями.

In [ ]:
vm = pd.read_csv("study2_vdjdb_matches.csv")
null = pd.read_csv("study2_vdjdb_null.csv")
print("Доля совпадений с VDJdb (мышиный TRA):")
print(null[['set','lev1_pct','exact_pct']].round(1).to_string(index=False))
enr_fold = null[null.set=="alloreactive_candidates"]["exact_pct"].iloc[0] / null[null.set=="g1_only_null"]["exact_pct"].iloc[0]
print(f"\nОбогащение по точным совпадениям над фоном g1: {enr_fold:.1f}x")
print("\nПреобладающие антигены у семейства NSNNRIF:")
print(vm[vm.get("motif","").astype(str)=="NSNNRIF"]["vdjdb_antigen"].value_counts().head(4).to_string())

**Вывод (раздел 9).** 21.1% высококонфиденциальных кандидатов точно совпадают с известными
антиген-специфичными мышиными TCR (против 2.5% у фона g1 — обогащение **8.4×**; перемешанный фон 0%).
Семейство NSNNRIF преимущественно картируется на эпитоп **ASNENMETM** (33 % совпадений,
далее VEALYLVCG — 22 %), семейство SSGSWQLI — на **KYNKANVFL** (21 %) — известные минорные
антигены гистосовместимости мыши. Это независимое подтверждение антигенной природы найденных клонов.

![VDJdb](figures_study2/fig_study2_vdjdb.png)

## 10. Интеграция сигналов → ранжированный список кандидатов

Все сигнальные слои объединяются в единый композитный балл аллореактивности
(ранг-нормализация каждого сигнала, взвешенная сумма):
- **алло-специфичность** по доле UMI (вес 2.0)
- **отсутствие у g1** (1.5)
- **воспроизводимость** у 10 алло-мышей (1.5)
- **клональная экспансия** (1.0)
- **членство в конвергентном мотиве** (1.0)

In [ ]:
cand = pd.read_csv("study2_alloreactive_candidates.csv")
hc = pd.read_csv("study2_high_confidence_candidates.csv")
print(f"Всего кандидатов (≥2 алло-мыши): {len(cand):,}")
print(f"Высококонфиденциальных (отсутствуют у g1, ≥8 алло-мышей): {(pd.read_parquet('study2_alloreactive_candidates_full.parquet').pipe(lambda d: (d.g1_mice==0)&(d.allo_mice>=8))).sum():,}")
print("\nТоп-15 ранжированных аллореактивных кандидатов:")
print(cand.head(15)[['rank','cdr3aa','v','j','motif','allo_mice','g1_mice','alloreactivity_score']].to_string(index=False))

**Итоговый вывод.** Интегрированное ранжирование выявляет чёткую конвергентную аллореактивную
сигнатуру: топ-кандидаты — это варианты мотивов **NSNNRIF/TRAJ31** (4 458 вариантов через 12+ TRAV)
и **SSGSWQLI/TRAJ22** (7 072 варианта), присутствующие у всех 10 аллогенных мышей и отсутствующие
у интактных. Один CDR3-мотив достигается множеством разных V-генов — классический признак
конвергентного антиген-обусловленного отбора. Мотив NSNNRIF достигается через множество V-генов с наибольшим числом вариантов у TRAV12-2 (161)
и TRAV6-6 (159); TRAV6-6 — один из кандидатов Исследования 1, что связывает результаты двух исследований.

![Интеграция](figures_study2/fig_study2_integration.png)